# Pipeline End-to-End — Node-by-Node Trace

Notebook chay toan bo pipeline EduBot, phan ro **tung node** de kiem soat:

```
1 ContextAnalyzer -> 2 IntentRouter (LLM) -> 3 SessionManager -> 4 ActionPlanner -> 5 RAG Search -> 6 Handler -> 7 Session Save
```

Ket qua pipeline duoc ghi ra `pipeline_trace.log` dang **JSON** (reset moi query). File `app.log` giu persistent log.

---
## 0. Setup — Path & Environment

In [ ]:
import sys
import os
import time
import json
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent.parent
print(f"Project root: {PROJECT_ROOT}")

# Add src to path
for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Load env
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

print(f"API Key: {'SET' if os.getenv('GENAI_API_KEY') else 'NOT SET'}")
print(f"Python: {sys.version.split()[0]}")

Project root: c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\ĐATN
API Key: SET
Python: 3.12.4


---
## 1. Init — CustomSearch + Reranker + Orchestrator

In [2]:
from src.config.config import settings
from src.rag.retrieve_rebuild import CustomSearch
from src.rag.reranker import Reranker
from src.llm.orchestrator import Orchestrator

DATA_DIR = PROJECT_ROOT / 'data'
CHUNKS_PATH = str(DATA_DIR / 'rag_chunks_v2.json')
EMBEDDINGS_PATH = str(DATA_DIR / 'embeddings.npy')

print("=" * 60)
print("[NODE 0] Initializing Components")
print("=" * 60)

# 1a. CustomSearch
t0 = time.time()
searcher = CustomSearch(chunks_path=CHUNKS_PATH, embeddings_path=EMBEDDINGS_PATH)
print(f"  CustomSearch: {searcher.corpus_size} chunks, dim={searcher.embeddings.shape[1]} ({time.time()-t0:.2f}s)")

# 1b. Reranker
reranker = Reranker()
print(f"  Reranker: {settings.RERANKER_MODEL} (lazy load)")

# 1c. Orchestrator
orch = Orchestrator(retriever=searcher, reranker=reranker)
print(f"  Orchestrator: ready")
print(f"  LLM Model: {settings.LLM_MODEL}")
print("=" * 60)

[00:51:20] INFO    | Tokenizing 2348 docs with underthesea...


[NODE 0] Initializing Components
CustomSearch initialized: 2348 docs, vocab=9672, avgdl=137.7
  CustomSearch: 2348 chunks, dim=768 (34.66s)
  Reranker: AITeamVN/Vietnamese_Reranker (lazy load)
  Orchestrator: ready
  LLM Model: gemini-2.5-flash-lite


---
## 2. Helper — Log Viewer & Node Runner

In [3]:
TRACE_LOG = PROJECT_ROOT / 'logs' / 'pipeline_trace.log'
APP_LOG = PROJECT_ROOT / 'logs' / 'app.log'

def show_trace_log():
    """Hien thi pipeline_trace.log (JSON format)."""
    if TRACE_LOG.exists():
        content = TRACE_LOG.read_text(encoding='utf-8')
        try:
            data = json.loads(content)
            print(json.dumps(data, indent=2, ensure_ascii=False, default=str))
        except json.JSONDecodeError:
            print(content)  # fallback to raw text
    else:
        print('[!] pipeline_trace.log chua ton tai')

def load_trace_json() -> dict:
    """Load pipeline_trace.log as dict."""
    if TRACE_LOG.exists():
        return json.loads(TRACE_LOG.read_text(encoding='utf-8'))
    return {}

def show_app_log(n=30):
    """Hien thi n dong cuoi cua app.log."""
    if APP_LOG.exists():
        lines = APP_LOG.read_text(encoding='utf-8').strip().split('\n')
        for line in lines[-n:]:
            print(line)
    else:
        print('[!] app.log chua ton tai')

def show_debug_info(debug_info: dict):
    """Hien thi debug info tu orchestrator.last_debug_info."""
    if not debug_info:
        print('[!] Chua co debug info')
        return
    print(json.dumps(debug_info, indent=2, ensure_ascii=False, default=str))

print('Helpers loaded: show_trace_log(), load_trace_json(), show_app_log(n), show_debug_info(info)')

Helpers loaded: show_trace_log(), load_trace_json(), show_app_log(n), show_debug_info(info)


---
## 3. Chay tung Node rieng le

Phan nay tach pipeline ra tung buoc de debug.

### 3.1 — Node: ContextAnalyzer

In [4]:
import json
from src.llm.context_analyzer import ContextAnalyzer

analyzer = ContextAnalyzer()
test_cases = [
    {"query": "Mạng LAN là gì?", "history": ""},
    {"query": "Nó khác gì với WAN?", "history": "user: Mạng LAN là gì?\nassistant: Mạng LAN là..."},
    {"query": "So sánh hai loại này", "history": "user: Giải thích hub và switch..."},
    {"query": "Thầy có thể nói rõ hơn được không?", "history": "user: Giao thức IP..."},
    {"query": "Vâng ạ", "history": "user: Em đã hiểu chưa?"},
    {"query": "Giải thích chi tiết về tầng mạng", "history": "user: Mô hình OSI..."},
]

output = []
for tc in test_cases:
    needs = analyzer.needs_contextualization(tc['query'], tc['history'])
    output.append({"query": tc['query'], "needs_context": needs})

print(json.dumps({"node": "ContextAnalyzer", "results": output}, indent=2, ensure_ascii=False))


{
  "node": "ContextAnalyzer",
  "results": [
    {
      "query": "Mạng LAN là gì?",
      "needs_context": false
    },
    {
      "query": "Nó khác gì với WAN?",
      "needs_context": true
    },
    {
      "query": "So sánh hai loại này",
      "needs_context": true
    },
    {
      "query": "Thầy có thể nói rõ hơn được không?",
      "needs_context": true
    },
    {
      "query": "Vâng ạ",
      "needs_context": false
    },
    {
      "query": "Giải thích chi tiết về tầng mạng",
      "needs_context": false
    }
  ]
}


### 3.2 — Node: IntentRouter (LLM Call)

In [5]:
import json, time
from src.llm.intent_router import IntentRouter

router = IntentRouter()
test_queries = [
    "tổng hợp kiến thức lớp 10 chủ đề 1 bộ sách cánh diều",
    "Giai thich TCP/IP la gi theo sach Ket Noi Tri Thuc",
    "Tao 3 cau trac nghiem ve mang may tinh (khong co sach)",
    "Xin chao ban la ai",
]

results = []
for q in test_queries:
    t0 = time.time()
    res = router.detect(query=q)
    results.append({"query": q, "intent": res.primary_intent, "topic": res.topic, "book": res.book, "time_s": round(time.time()-t0, 2)})

print(json.dumps({"node": "IntentRouter", "results": results}, indent=2, ensure_ascii=False))


[00:52:33] INFO    | IntentRouter: intent=explain, task_type=None, topic=Chủ đề 1 - Lớp 10, is_new_topic=True, book=CD
[00:52:33] INFO    | IntentRouter: intent=explain, task_type=None, topic=None, is_new_topic=True, book=KNTT
[00:52:37] INFO    | IntentRouter: intent=generate, task_type=mcq, topic=None, is_new_topic=True, book=None
[00:52:38] INFO    | IntentRouter: intent=chat, task_type=None, topic=None, is_new_topic=False, book=None


{
  "node": "IntentRouter",
  "results": [
    {
      "query": "tổng hợp kiến thức lớp 10 chủ đề 1 bộ sách cánh diều",
      "intent": "explain",
      "topic": "Chủ đề 1 - Lớp 10",
      "book": "CD",
      "time_s": 1.46
    },
    {
      "query": "Giai thich TCP/IP la gi theo sach Ket Noi Tri Thuc",
      "intent": "explain",
      "topic": null,
      "book": "KNTT",
      "time_s": 0.82
    },
    {
      "query": "Tao 3 cau trac nghiem ve mang may tinh (khong co sach)",
      "intent": "generate",
      "topic": null,
      "book": null,
      "time_s": 3.19
    },
    {
      "query": "Xin chao ban la ai",
      "intent": "chat",
      "topic": null,
      "book": null,
      "time_s": 1.41
    }
  ]
}


### 3.3 — Node: SessionManager

In [6]:
import json
from src.llm.intent_router import IntentResult
from src.llm.session_manager import SessionManager
from src.llm.session_store import SessionStore
from src.llm.memory import MemoryManager

memory = MemoryManager()
store = SessionStore(storage_path=str(PROJECT_ROOT / 'data' / 'sessions'))
sm = SessionManager(session_store=store, memory=memory)

intents = [
    IntentResult(primary_intent="chat", topic=None, is_new_topic=True),
    IntentResult(primary_intent="generate", task_type="mcq", topic="Mang may tinh", is_new_topic=True, book="CD"),
]

sessions = []
for intent in intents:
    s = sm.resolve_session(intent)
    sessions.append({"session_id": s.session_id, "topic": s.topic, "book": s.book})

print(json.dumps({"node": "SessionManager", "resolved_sessions": sessions}, indent=2, ensure_ascii=False))


[00:52:38] INFO    | No current session, creating new
[00:52:38] INFO    | New session created: id=18415219, topic='', intent=chat, book=None
[00:52:38] INFO    | Topic changed: '' -> 'Mang may tinh', creating new session
[00:52:38] INFO    | New session created: id=13da82e9, topic='Mang may tinh', intent=generate, book=CD


{
  "node": "SessionManager",
  "resolved_sessions": [
    {
      "session_id": "18415219",
      "topic": "",
      "book": null
    },
    {
      "session_id": "13da82e9",
      "topic": "Mang may tinh",
      "book": "CD"
    }
  ]
}


### 3.4 — Node: ActionPlanner

In [7]:
import json
from src.llm.action_planner import ActionPlanner
from src.llm.intent_router import IntentResult
from src.llm.memory import Session

planner = ActionPlanner()
scenarios = [
    (IntentResult(primary_intent="generate", task_type="mcq"), None, "Tao 3 cau MCQ"),
    (IntentResult(primary_intent="interact"), Session(session_id="S2", topic="Network", intent="generate"), "ôn lại câu sai"),
]

results = []
for intent, sess, msg in scenarios:
    plan = planner.plan(intent, sess, msg)
    results.append({"query": msg, "action": plan.action.value, "reason": plan.reason})

print(json.dumps({"node": "ActionPlanner", "results": results}, indent=2, ensure_ascii=False))


{
  "node": "ActionPlanner",
  "results": [
    {
      "query": "Tao 3 cau MCQ",
      "action": "generate_quiz",
      "reason": "task_type=mcq"
    },
    {
      "query": "ôn lại câu sai",
      "action": "review_wrong",
      "reason": "Review keywords detected, round_id=None"
    }
  ]
}


### 3.5 — Node: RAG Search (BM25 + Semantic + RRF -> Reranker)

In [8]:
import json, time
query = "Mang may tinh la gi và có những loại nào"
t0 = time.time()
search_results = searcher.search(query, top_k=5)
res = reranker.rerank(query, search_results, top_n=3)

print(json.dumps({
    "node": "RAG_Search",
    "query": query,
    "time_s": round(time.time()-t0, 2),
    "results": [{"score": round(r.get('rerank_score', 0), 4), "content": r['content'][:100]} for r in res]
}, indent=2, ensure_ascii=False))


Loading embedding model: dangvantuan/vietnamese-document-embedding...
Model loaded on cuda
Loading reranker: AITeamVN/Vietnamese_Reranker...
Reranker loaded on cuda
{
  "node": "RAG_Search",
  "query": "Mang may tinh la gi và có những loại nào",
  "time_s": 23.23,
  "results": [
    {
      "score": 0.0001,
      "content": "Xét các bài toán sau:\n*   1) Hệ thống thư điện tử cần xác định các email nghi là thư rác và đánh dấu"
    },
    {
      "score": 0.0001,
      "content": "Mệnh đề là một khẳng định có tính chất hoặc đúng hoặc sai. Ví dụ “Hà Nội là Thủ đô của Việt Nam” là "
    },
    {
      "score": 0.0,
      "content": "**Năm nhuận** là những năm chia hết cho 400 hoặc là những năm chia hết cho 4 nhưng không chia hết ch"
    }
  ]
}


### 3.5b — Node: Multi-Query RAG Search (New Feature)
Thu nghiem gui query toi QueryRewriter de ra 2-3 queries khao sat do phu RAG, giong luong thuc te cua Orchestrator.

In [9]:
import json
from src.llm.query_rewriter import QueryRewriter

rewriter = QueryRewriter()
query = "ưu điểm của nó?"
context = "user: Mạng LAN là gì? assistant: Mạng LAN là mạng máy tính cục bộ..."

# 1. Rewrite Query
rewritten_queries = rewriter.rewrite(query, context)

# 2. RAG Search voi da truy van
all_chunks = []
seen = set()
for q in rewritten_queries:
    results = searcher.search(q, top_k=5)
    for r in results:
        if r["doc_id"] not in seen:
            seen.add(r["doc_id"])
            all_chunks.append(r)

# 3. Rerank
reranked_mq = reranker.rerank(rewritten_queries[0] if rewritten_queries else query, all_chunks, top_n=5)

# 4. In ra JSON thay vi text raw
output_data = {
    "node": "3.5b Multi-Query RAG Search",
    "inputs": {
        "original_query": query,
        "history_context": context
    },
    "rewrite_phase": {
        "num_queries_generated": len(rewritten_queries),
        "rewritten_queries": rewritten_queries
    },
    "retrieval_phase": {
        "total_chunks_collected": len(all_chunks),
        "unique_chunks_deduplicated": len(all_chunks)
    },
    "rerank_phase": {
        "primary_query_for_rerank": rewritten_queries[0] if rewritten_queries else query,
        "top_n_after_rerank": len(reranked_mq),
        "top_chunks": [
            {
                "rank": idx + 1,
                "doc_id": r.get("doc_id", "unknown"),
                "score": round(r.get("rerank_score", 0), 4),
                "metadata": r.get("metadata", {}),
                "content_preview": r["content"][:200].replace("\n", " ") + "..."
            }
            for idx, r in enumerate(reranked_mq)
        ]
    }
}

print("=" * 60)
print("[NODE 5b] Multi-Query RAG Search (JSON Structured Output)")
print("=" * 60)
print(json.dumps(output_data, indent=2, ensure_ascii=False))


[00:53:08] INFO    | QueryRewriter: needs_rewrite=True, queries=['Ưu điểm của mạng LAN là gì?', 'Lợi ích của mạng cục bộ LAN', 'Điểm mạnh của mạng LAN']


[NODE 5b] Multi-Query RAG Search (JSON Structured Output)
{
  "node": "3.5b Multi-Query RAG Search",
  "inputs": {
    "original_query": "ưu điểm của nó?",
    "history_context": "user: Mạng LAN là gì? assistant: Mạng LAN là mạng máy tính cục bộ..."
  },
  "rewrite_phase": {
    "num_queries_generated": 3,
    "rewritten_queries": [
      "Ưu điểm của mạng LAN là gì?",
      "Lợi ích của mạng cục bộ LAN",
      "Điểm mạnh của mạng LAN"
    ]
  },
  "retrieval_phase": {
    "total_chunks_collected": 8,
    "unique_chunks_deduplicated": 8
  },
  "rerank_phase": {
    "primary_query_for_rerank": "Ưu điểm của mạng LAN là gì?",
    "top_n_after_rerank": 5,
    "top_chunks": [
      {
        "rank": 1,
        "doc_id": 63,
        "score": 0.6059,
        "metadata": {
          "book": "CD",
          "grade": "10",
          "topic": "B",
          "topic_name": "Mạng máy tính và Internet – Internet hôm nay và ngày mai",
          "lesson": "Bài 2",
          "lesson_name": "ĐIỆN TOÁN ĐÁ

### 3.6 — Node: Handler (LLM Generation)

In [10]:
from src.llm.handlers.chat_handler import ChatHandler
from src.llm.handlers.explain_handler import ExplainHandler
from src.llm.handlers.question.mcq_handler import MCQHandler
from src.llm.utils import format_contexts
import time, json

context_text = format_contexts(reranked)  # From previous cell

chat_h = ChatHandler()
t0 = time.time()
chat_resp = chat_h.handle(query='Mang may tinh la gi?', context=context_text)
chat_time = time.time() - t0

output_data_chat = {
    'node': '6a. ChatHandler',
    'metrics': {
        'execution_time_s': round(chat_time, 2),
        'response_length_chars': len(chat_resp)
    },
    'response_preview': chat_resp[:200] + '...'
}
print('=' * 60)
print('[NODE 6a] ChatHandler (JSON Structured Output)')
print('=' * 60)
print(json.dumps(output_data_chat, indent=2, ensure_ascii=False))


NameError: name 'reranked' is not defined

In [ ]:
explain_h = ExplainHandler()
t0 = time.time()
explain_resp = explain_h.handle(query='Giai thich mang LAN khac gi mang WAN', context=context_text)
explain_time = time.time() - t0

output_data_explain = {
    'node': '6b. ExplainHandler',
    'metrics': {
        'execution_time_s': round(explain_time, 2),
        'response_length_chars': len(explain_resp)
    },
    'response_preview': explain_resp[:200] + '...'
}
print('=' * 60)
print('[NODE 6b] ExplainHandler (JSON Structured Output)')
print('=' * 60)
print(json.dumps(output_data_explain, indent=2, ensure_ascii=False))


--- ExplainHandler ---
  Time: 3.16s | Length: 4128
  Preview: Chào em, thầy là EduBot, trợ lý học tập Tin học THPT của em đây! Hôm nay chúng ta sẽ cùng tìm hiểu về sự khác biệt giữa mạng LAN và mạng WAN nhé. Đây là hai khái niệm rất quan trọng trong lĩnh vực mạng máy tính.

### 1. Khái niệm cốt lõi

*   **Mạng LAN (Local Area Network)**: Là mạng kết nối các th



In [ ]:
mcq_h = MCQHandler()
t0 = time.time()
mcq_resp = mcq_h.handle(topic='Mang may tinh', context=context_text, num_questions=3)
mcq_time = time.time() - t0

output_data_mcq = {
    'node': '6c. MCQHandler',
    'metrics': {
        'execution_time_s': round(mcq_time, 2),
        'questions_generated': len(mcq_resp) if isinstance(mcq_resp, list) else 0
    },
    'questions_preview': [q.get('question_text') for q in mcq_resp][:2] if isinstance(mcq_resp, list) else []
}
print('=' * 60)
print('[NODE 6c] MCQHandler (JSON Structured Output)')
print('=' * 60)
print(json.dumps(output_data_mcq, indent=2, ensure_ascii=False))


--- MCQHandler ---
  Time: 1.49s
  Questions: 3
  Display:
Câu hỏi 1:
Trong lập trình, đại lượng nào chỉ nhận giá trị Đúng hoặc Sai?

A. Đại lượng số nguyên
B. Đại lượng số thực
C. Đại lượng lôgic
D. Đại lượng chuỗi ký tự

________________________________________

Câu hỏi 2:
Năm nào sau đây KHÔNG phải là năm nhuận theo quy tắc chia hết cho 4 và chia hết cho 100?

A. 2000
B. 1900
C. 2024
D. 1600

________________________________________

Câu hỏi 3:
Theo Context 4, giá trị lôgic 'Đúng' và 'Sai' thường được biểu diễn tương ứng lần lượt là bao nhiêu?

A. 1 


### 3.7 — Node: Question Validator

In [ ]:
from src.llm.validators.question_validator import QuestionValidator

if mcq_result:
    print("=" * 60)
    print("[NODE 7] Question Validator")
    print("=" * 60)
    
    validator = QuestionValidator()
    t0 = time.time()
    val_result = validator.validate(
        question_type="mcq",
        context=context_text,
        questions_json=json.dumps(mcq_result.model_dump())
    )
    val_time = time.time() - t0
    
    print(f"  Time: {val_time:.2f}s")
    print(f"  All valid: {val_result.all_valid}")
    print(f"  Approved: {len(val_result.approved_questions)}")
    print(f"  Validations: {len(val_result.validations)}")
    for v in val_result.validations:
        print(f"    - {v}")
else:
    print("[!] Skip validator -- mcq_result is None")

[NODE 7] Question Validator
  Time: 2.75s
  All valid: True
  Approved: 3
  Validations: 3
    - index=1 is_valid=True issues=[] fixed_question=None
    - index=2 is_valid=True issues=[] fixed_question=None
    - index=3 is_valid=True issues=[] fixed_question=None


---
## 4. Full Pipeline — Orchestrator.ask()

Chay toan bo pipeline end-to-end. Ket qua trace duoc ghi vao `pipeline_trace.log` dang **JSON**.

In [ ]:
import json, time
orch.memory = MemoryManager()
orch.session_manager.memory = orch.memory
QUERY = "tổng hợp kiến thức tin học của lơp 12"

t0 = time.time()
response_chunks = []
for chunk in orch.ask(QUERY, ui_book="CD"):
    response_chunks.append(chunk)

print(json.dumps({
    "node": "Full_Pipeline",
    "query": QUERY,
    "total_time_s": round(time.time()-t0, 2),
    "debug_info": orch.last_debug_info,
    "response_preview": "".join(response_chunks)[:500] + "..."
}, indent=2, ensure_ascii=False))


[00:38:39] INFO    | ============================================================
[00:38:39] INFO    | QUERY: 'tổng hợp kiến thức tin học của lơp 12'


FULL PIPELINE -- Query: 'tổng hợp kiến thức tin học của lơp 12'


[00:38:40] INFO    | IntentRouter: intent=explain, task_type=None, topic=kiến thức tin học lớp 12, is_new_topic=True, book=None
[00:38:40] INFO    | IntentRouter (1.47s): intent=explain, task_type=None, topic=kiến thức tin học lớp 12, is_new_topic=True
[00:38:40] INFO    | Topic changed: 'null' -> 'kiến thức tin học lớp 12', creating new session
[00:38:40] INFO    | New session created: id=1b0f45d1, topic='kiến thức tin học lớp 12', intent=explain, book=None
[00:38:40] INFO    | Session: id=1b0f45d1, topic='kiến thức tin học lớp 12', msgs=0
[00:38:40] INFO    | ActionPlan: explain_concept (General concept explanation)
[00:38:40] INFO    | Book: ui=CD, llm=None, session=CD -> effective=CD
[00:38:40] INFO    | RAGAgent: book filter='CD' → 1204 chunks in scope
[00:38:40] INFO    | RAGAgent: strategy=broad | grade=12 | topic=kiến thức tin học lớp 12 | book=CD | Query tổng quát: broad=True, grade_only=True, topic_broad=True
[00:38:40] INFO    | RAGAgent done: 29 chunks, 0.00s
[00:39:27] INF


RESPONSE (43867 chars, 48.45s):
Dang tim tai lieu de giai thich...

Chào em, thầy là EduBot, trợ lý học tập Tin học THPT của em đây! Thầy rất vui được cùng em ôn tập lại kiến thức Tin học lớp 12. Lớp 12 có rất nhiều chủ đề thú vị và quan trọng, từ những khái niệm nền tảng đến những công nghệ hiện đại. Chúng ta sẽ cùng nhau đi qua từng phần nhé!

Để giúp em hệ thống hóa kiến thức một cách tốt nhất, thầy sẽ trình bày theo cấu trúc mà em đã yêu cầu: **Khái niệm cốt lõi**, **Giải thích chi tiết**, **Ví dụ minh họa**, **So sánh (nếu phù hợp)** và **Tóm tắt**.

Chúng ta bắt đầu với những chủ đề chính nhé!

---

### **1. Trí tuệ nhân tạo (AI) và Học máy**

**1.1. Trí tuệ nhân tạo (AI)**

*   **Khái niệm cốt lõi**: Trí tuệ nhân tạo (AI) là lĩnh vực khoa học máy tính tập trung vào việc tạo ra các hệ thống có khả năng thực hiện các nhiệm vụ mà thông thường đòi hỏi trí tuệ con người, như học hỏi, suy luận, giải quyết vấn đề, nhận thức và hiểu ngôn ngữ.

*   **Giải thích chi tiết**:
    *   **Khá

### 4.1 — Xem Pipeline Trace JSON

In [ ]:
print("=" * 70)
print("PIPELINE TRACE (pipeline_trace.log — JSON)")
print("=" * 70)
show_trace_log()

PIPELINE TRACE (pipeline_trace.log — JSON)
{
  "query": "tổng hợp kiến thức tin học của lơp 12",
  "timestamp": "2026-04-12 00:37:03",
  "steps": [
    {
      "node": "ContextAnalyzer",
      "enriched": false,
      "rewrite": null
    },
    {
      "node": "IntentRouter",
      "primary_intent": "explain",
      "task_type": null,
      "topic": "kiến thức tin học lớp 12",
      "is_new_topic": true,
      "book": null,
      "time_s": 1.5
    },
    {
      "node": "SessionManager",
      "session_id": "67dd0b4a",
      "topic": "kiến thức tin học lớp 12",
      "intent": "explain",
      "book": null,
      "total_messages": 0,
      "has_quiz_state": false,
      "has_slide_state": false
    },
    {
      "node": "ActionPlanner",
      "action": "explain_concept",
      "reason": "General concept explanation",
      "round_id": null
    },
    {
      "node": "RAG",
      "queries_used": [
        "tổng hợp kiến thức tin học của lơp 12"
      ],
      "strategy": "broad",
     

### 4.2 — Xem Debug Info (orchestrator.last_debug_info)

In [ ]:
print("=" * 70)
print("DEBUG INFO (orchestrator.last_debug_info)")
print("=" * 70)
show_debug_info(orch.last_debug_info)

DEBUG INFO (orchestrator.last_debug_info)
{
  "query": "tổng hợp kiến thức tin học của lơp 12",
  "timestamp": "2026-04-12 00:37:03",
  "steps": [
    {
      "node": "ContextAnalyzer",
      "enriched": false,
      "rewrite": null
    },
    {
      "node": "IntentRouter",
      "primary_intent": "explain",
      "task_type": null,
      "topic": "kiến thức tin học lớp 12",
      "is_new_topic": true,
      "book": null,
      "time_s": 1.5
    },
    {
      "node": "SessionManager",
      "session_id": "67dd0b4a",
      "topic": "kiến thức tin học lớp 12",
      "intent": "explain",
      "book": null,
      "total_messages": 0,
      "has_quiz_state": false,
      "has_slide_state": false
    },
    {
      "node": "ActionPlanner",
      "action": "explain_concept",
      "reason": "General concept explanation",
      "round_id": null
    },
    {
      "node": "RAG",
      "queries_used": [
        "tổng hợp kiến thức tin học của lơp 12"
      ],
      "strategy": "broad",
      

### 4.3 — Xem App Log (n dong cuoi)

In [ ]:
print("=" * 70)
print("APP LOG (last 20 lines)")
print("=" * 70)
show_app_log(20)

APP LOG (last 20 lines)
[2026-04-12 00:36:08] DEBUG   | chatbot.session_store | Session saved: 9339a596 -> c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\ĐATN\data\sessions\9339a596.json
[2026-04-12 00:36:08] DEBUG   | chatbot.session_manager | Session archived: 9339a596
[2026-04-12 00:36:08] INFO    | chatbot.session_manager | New session created: id=41c3925e, topic='Mang may tinh', intent=generate, book=CD
[2026-04-12 00:36:08] DEBUG   | chatbot.session_manager | Keeping current session: 41c3925e
[2026-04-12 00:36:50] INFO    | chatbot.query_rewriter | QueryRewriter: needs_rewrite=True, queries=['Ưu điểm của mạng LAN là gì?', 'Lợi ích của mạng cục bộ LAN', 'Điểm mạnh của mạng LAN']
[2026-04-12 00:37:03] INFO    | chatbot | ============================================================
[2026-04-12 00:37:03] INFO    | chatbot | QUERY: 'tổng hợp kiến thức tin học của lơp 12'
[2026-04-12 00:37:05] INFO    | chatbot.intent_router | IntentRouter: intent=explain,

---
## 4.5 — Book Blocking Test

Kiem tra Orchestrator block khi query can tao RAG (generate/explain) nhung khong co book.

In [ ]:
orch.memory = MemoryManager()
orch.session_manager.memory = orch.memory

QUERY = "Tao 3 cau hoi ve tri tue nhan tao"
print("=" * 70)
print(f"BOOK BLOCKING TEST -- Query: '{QUERY}'")
print("=" * 70)

block_resp = []
for chunk in orch.ask(QUERY, ui_book=None):
    block_resp.append(chunk)

print("\nRESPONSE:\n")
print("".join(block_resp))
print("\nDEBUG INFO:\n")
show_debug_info(orch.last_debug_info)


[00:37:07] INFO    | ============================================================
[00:37:07] INFO    | QUERY: 'Tao 3 cau hoi ve tri tue nhan tao'


BOOK BLOCKING TEST -- Query: 'Tao 3 cau hoi ve tri tue nhan tao'


[00:37:09] INFO    | IntentRouter: intent=generate, task_type=essay, topic=trí tuệ nhân tạo, is_new_topic=True, book=None
[00:37:09] INFO    | IntentRouter (2.06s): intent=generate, task_type=essay, topic=trí tuệ nhân tạo, is_new_topic=True
[00:37:09] INFO    | Topic changed: 'kiến thức tin học lớp 12' -> 'trí tuệ nhân tạo', creating new session
[00:37:09] INFO    | New session created: id=2a603f61, topic='trí tuệ nhân tạo', intent=generate, book=None
[00:37:09] INFO    | Session: id=2a603f61, topic='trí tuệ nhân tạo', msgs=0
[00:37:09] INFO    | ActionPlan: generate_quiz (task_type=essay)
[00:37:09] INFO    | Book: ui=None, llm=None, session=None -> effective=None



RESPONSE:

📚 Hệ thống hỗ trợ 2 bộ sách SGK Tin học THPT:
- **Cánh Diều** (CD)
- **Kết Nối Tri Thức** (KNTT)

Vui lòng cho mình biết bạn đang học theo bộ sách nào để mình tra cứu chính xác nhất nhé! 🎯

DEBUG INFO:

{
  "query": "Tao 3 cau hoi ve tri tue nhan tao",
  "timestamp": "2026-04-12 00:37:07",
  "steps": [
    {
      "node": "ContextAnalyzer",
      "enriched": false,
      "rewrite": null
    },
    {
      "node": "IntentRouter",
      "primary_intent": "generate",
      "task_type": "essay",
      "topic": "trí tuệ nhân tạo",
      "is_new_topic": true,
      "book": null,
      "time_s": 2.06
    },
    {
      "node": "SessionManager",
      "session_id": "2a603f61",
      "topic": "trí tuệ nhân tạo",
      "intent": "generate",
      "book": null,
      "total_messages": 0,
      "has_quiz_state": false,
      "has_slide_state": false
    },
    {
      "node": "ActionPlanner",
      "action": "generate_quiz",
      "reason": "task_type=essay",
      "round_id": null
   

---
## 5. Multi-Query Test — Chuoi hoi thoai

Test pipeline lien tiep nhieu query de kiem tra session management.

In [ ]:
import json, time
orch.memory = MemoryManager()
orch.session_manager.memory = orch.memory
queries = ["Xin chao", "Giai thich mang may tinh", "Tao 3 cau trac nghiem"]

history_results = []
for q in queries:
    list(orch.ask(q, ui_book="KNTT"))
    history_results.append(orch.last_debug_info)

print(json.dumps({"node": "Multi_Query_Sequence", "history": history_results}, indent=2, ensure_ascii=False))


[00:37:09] INFO    | ============================================================
[00:37:09] INFO    | QUERY: 'Xin chao ban la ai'


MULTI-QUERY TEST

[Query 1/3]: Xin chao ban la ai


[00:37:11] INFO    | IntentRouter: intent=chat, task_type=None, topic=None, is_new_topic=True, book=None
[00:37:11] INFO    | IntentRouter (2.15s): intent=chat, task_type=None, topic=None, is_new_topic=True
[00:37:11] INFO    | Session: id=2a603f61, topic='trí tuệ nhân tạo', msgs=2
[00:37:11] INFO    | ActionPlan: chat (Default chat intent)
[00:37:11] INFO    | Book: ui=KNTT, llm=None, session=KNTT -> effective=KNTT
[00:37:11] INFO    | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[00:37:11] INFO    | RAGAgent: strategy=standard | grade=None | topic=None | book=KNTT | Query cụ thể, không có grade/topic context
[00:37:36] INFO    | RAGAgent done: 5 chunks, 24.57s
[00:37:37] INFO    | Total time: 28.02s
[00:37:37] INFO    | ============================================================
[00:37:37] INFO    | ============================================================
[00:37:37] INFO    | QUERY: 'Giai thich mang may tinh cho toi'


  [1] ContextAnalyzer: enriched=False
  [2] IntentRouter: intent=chat | topic=None | time=2.15s
  [3] SessionManager: id=2a603f61 | topic=trí tuệ nhân tạo | msgs=2
  [4] ActionPlanner: action=chat | reason=Default chat intent
  [5] RAG: search=None | reranked=None | time=24.57s
  [6] Handler: action=chat | status=success | {"rag_chunks": 5, "chat_time_s": 1.3, "response_length": 197}
  Total: 28.03s | Response: 197 chars
  Preview: Chào bạn! 👋 Mình là EduBot, trợ lý học tập Tin học THPT Việt Nam. Mình ở đây để giúp bạn học tốt môn Tin học theo sách giáo khoa.

Bạn có câu hỏi gì về Tin học THPT không? Mình sẵn sàng giải đáp! 😊

[Query 2/3]: Giai thich mang may tinh cho toi


[00:37:39] INFO    | IntentRouter: intent=explain, task_type=None, topic=mạng máy tính, is_new_topic=True, book=None
[00:37:39] INFO    | IntentRouter (2.18s): intent=explain, task_type=None, topic=mạng máy tính, is_new_topic=True
[00:37:39] INFO    | Topic changed: 'trí tuệ nhân tạo' -> 'mạng máy tính', creating new session
[00:37:39] INFO    | New session created: id=a9fa3b5d, topic='mạng máy tính', intent=explain, book=None
[00:37:39] INFO    | Session: id=a9fa3b5d, topic='mạng máy tính', msgs=0
[00:37:39] INFO    | ActionPlan: explain_concept (General concept explanation)
[00:37:39] INFO    | Book: ui=KNTT, llm=None, session=KNTT -> effective=KNTT
[00:37:39] INFO    | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[00:37:39] INFO    | RAGAgent: strategy=broad | grade=None | topic=mạng máy tính | book=KNTT | Query tổng quát: broad=False, grade_only=False, topic_broad=True
[00:37:39] INFO    | RAGAgent done: 6 chunks, 0.01s
[00:37:45] INFO    | Total time: 7.90s
[00:37:45] INFO 

  [1] ContextAnalyzer: enriched=False
  [2] IntentRouter: intent=explain | topic=mạng máy tính | time=2.18s
  [3] SessionManager: id=a9fa3b5d | topic=mạng máy tính | msgs=0
  [4] ActionPlanner: action=explain_concept | reason=General concept explanation
  [5] RAG: search=None | reranked=None | time=0.01s
  [6] Handler: action=explain_concept | status=success | {"rag_chunks": 6, "explain_time_s": 5.68, "response_length": 4237}
  Total: 7.90s | Response: 4273 chars
  Preview: Dang tim tai lieu de giai thich...

Chào em, thầy là EduBot, trợ lý học tập Tin học THPT của em đây! Hôm nay chúng ta sẽ cùng nhau tìm hiểu về "mạng máy tính" nhé. Đây là một khái niệm rất quan trọng 

[Query 3/3]: Tao 3 cau trac nghiem ve chu de nay


[00:37:46] INFO    | IntentRouter: intent=generate, task_type=mcq, topic=null, is_new_topic=True, book=None
[00:37:46] INFO    | IntentRouter (1.04s): intent=generate, task_type=mcq, topic=null, is_new_topic=True
[00:37:46] INFO    | Topic changed: 'mạng máy tính' -> 'null', creating new session
[00:37:46] INFO    | New session created: id=aa3ed2e5, topic='null', intent=generate, book=None
[00:37:46] INFO    | Session: id=aa3ed2e5, topic='null', msgs=0
[00:37:46] INFO    | ActionPlan: generate_quiz (task_type=mcq)
[00:37:46] INFO    | Book: ui=KNTT, llm=None, session=KNTT -> effective=KNTT
[00:37:46] INFO    | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[00:37:46] INFO    | RAGAgent: strategy=hierarchical | grade=None | topic=null | book=KNTT | Query cụ thể + context (grade=None, topic=True) → HRAG
[00:37:46] INFO    | HRAG Phase 1: 594 parents searched → 3 selected → 3 unique lessons
[00:37:46] INFO    | HRAG Phase 2: scoped search on 19 child chunks
[00:37:54] INFO    | RAGAg

KeyboardInterrupt: 

### 5.1 — Xem Trace JSON cua query cuoi

In [ ]:
show_trace_log()

{
  "query": "Tao 3 cau trac nghiem ve chu de nay",
  "timestamp": "2026-04-09 02:02:41",
  "steps": [
    {
      "node": "ContextAnalyzer",
      "enriched": false
    },
    {
      "node": "IntentRouter",
      "primary_intent": "generate",
      "task_type": "mcq",
      "topic": null,
      "is_new_topic": true,
      "time_s": 0.88
    },
    {
      "node": "SessionManager",
      "session_id": "66342982",
      "topic": "mạng máy tính",
      "intent": "explain",
      "total_messages": 2,
      "has_quiz_state": false,
      "has_slide_state": false
    },
    {
      "node": "ActionPlanner",
      "action": "generate_quiz",
      "reason": "task_type=mcq",
      "round_id": null
    },
    {
      "node": "RAG",
      "strategy": "standard",
      "chunks_returned": 5,
      "time_s": 44.8,
      "filter": {
        "grade": null,
        "topic": null
      },
      "reason": "Query cụ thể"
    },
    {
      "node": "Handler",
      "handler": "MCQHandler",
      "action":

---
## 6. Custom Query — Tu nhap de test

In [ ]:
# === THAY DOI QUERY TAI DAY ===
CUSTOM_QUERY = "Tao 3 cau dung sai ve an toan thong tin"
# ===============================

print(f"Query: '{CUSTOM_QUERY}'")
print("=" * 70)

t0 = time.time()
chunks = list(orch.ask(CUSTOM_QUERY))
total = time.time() - t0
response = "".join(chunks)

print(f"\nResponse ({len(response)} chars, {total:.2f}s):")
print("=" * 70)
print(response[:1500])

print("\n" + "=" * 70)
print("PIPELINE TRACE JSON:")
print("=" * 70)
show_trace_log()

[02:03:32] INFO    | ============================================================
[02:03:32] INFO    | QUERY: 'Tao 3 cau dung sai ve an toan thong tin'


Query: 'Tao 3 cau dung sai ve an toan thong tin'


[02:03:33] INFO    | IntentRouter: intent=generate, task_type=true_false, topic=an toan thong tin, is_new_topic=True
[02:03:33] INFO    | IntentRouter (0.93s): intent=generate, task_type=true_false, topic=an toan thong tin, is_new_topic=True
[02:03:33] INFO    | Topic changed: 'mạng máy tính' -> 'an toan thong tin', creating new session
[02:03:33] INFO    | New session created: id=bbfd614a, topic='an toan thong tin', intent=generate
[02:03:33] INFO    | Session: id=bbfd614a, topic='an toan thong tin', msgs=0
[02:03:33] INFO    | ActionPlan: generate_quiz (task_type=true_false)
[02:03:33] INFO    | RAGAgent: strategy=standard | grade=None | topic=an toan thong tin | Query cụ thể
[02:04:19] INFO    | RAGAgent done: 5 chunks, 45.85s
[02:04:19] INFO    | RAG Search: 5 chunks (45.86s)
[02:04:19] INFO    | Generate: type=true_false, num=3
[02:04:21] INFO    | Handler.handle() -> 2.11s (attempt 1)
[02:04:23] INFO    | Validator: all_valid=True, approved=3 (2.10s)
[02:04:23] INFO    | Saved 3 


Response (628 chars, 51.02s):
Dang tim kiem tai lieu lien quan...Dang soan 3 cau hoi TRUE_FALSE...Dang kiem duyet chat luong...

❓ Câu 1:
   Switch sử dụng địa chỉ IP để chuyển tiếp các gói dữ liệu giữa các thiết bị trong mạng cục bộ.
   → Đúng hay Sai?

________________________________________

❓ Câu 2:
   Việc chia sẻ thông tin bịa đặt về dịch bệnh trên mạng xã hội có thể vi phạm pháp luật.
   → Đúng hay Sai?

________________________________________

❓ Câu 3:
   Bảng định tuyến (routing table) trong Router chứa địa chỉ MAC của các thiết bị trong mạng cục bộ.
   → Đúng hay Sai?

________________________________________


[Round 1] Da tao 3 cau hoi.

PIPELINE TRACE JSON:
{
  "query": "Tao 3 cau dung sai ve an toan thong tin",
  "timestamp": "2026-04-09 02:03:32",
  "steps": [
    {
      "node": "ContextAnalyzer",
      "enriched": false
    },
    {
      "node": "IntentRouter",
      "primary_intent": "generate",
      "task_type": "true_false",
      "topic": "an toan thong tin",


---
## 7. Timing Benchmark — Do thoi gian tung node

In [ ]:
# Reset memory
orch.memory = MemoryManager()
orch.session_manager.memory = orch.memory

benchmark_query = "Tao 3 cau trac nghiem ve he dieu hanh"

print(f"Benchmark: '{benchmark_query}'")
print("=" * 70)

chunks = list(orch.ask(benchmark_query))

# Load JSON trace
trace_data = load_trace_json()
steps = trace_data.get('steps', [])

print(f"\n{'Node':<25} {'Time (s)':>10}")
print('=' * 37)

for step in steps:
    node = step.get('node', '?')
    # Find timing field
    t = step.get('time_s') or step.get('generation_time_s') or step.get('chat_time_s') or step.get('explain_time_s') or step.get('slide_time_s') or step.get('scorer_time_s')
    label = node
    if node == 'Handler':
        label = f"Handler:{step.get('action','?')}"
    if t is not None:
        bar = '#' * int(t * 2)
        print(f"  {label:<23} {t:>8.2f}s  {bar}")
    else:
        print(f"  {label:<23}      -")

total = trace_data.get('total_time_s', 0)
print('=' * 37)
print(f"  {'TOTAL':<23} {total:>8.2f}s")

# Response info
resp = trace_data.get('response', {})
print(f"\nResponse: {resp.get('length', 0)} chars")

[02:04:23] INFO    | ============================================================
[02:04:23] INFO    | QUERY: 'Tao 3 cau trac nghiem ve he dieu hanh'


Benchmark: 'Tao 3 cau trac nghiem ve he dieu hanh'


[02:04:24] INFO    | IntentRouter: intent=generate, task_type=mcq, topic=hệ điều hành, is_new_topic=True
[02:04:24] INFO    | IntentRouter (0.97s): intent=generate, task_type=mcq, topic=hệ điều hành, is_new_topic=True
[02:04:24] INFO    | Topic changed: 'an toan thong tin' -> 'hệ điều hành', creating new session
[02:04:24] INFO    | New session created: id=5e35c39a, topic='hệ điều hành', intent=generate
[02:04:24] INFO    | Session: id=5e35c39a, topic='hệ điều hành', msgs=0
[02:04:24] INFO    | ActionPlan: generate_quiz (task_type=mcq)
[02:04:24] INFO    | RAGAgent: strategy=standard | grade=None | topic=hệ điều hành | Query cụ thể
[02:04:53] INFO    | RAGAgent done: 5 chunks, 29.14s
[02:04:53] INFO    | RAG Search: 5 chunks (29.15s)
[02:04:53] INFO    | Generate: type=mcq, num=3
[02:04:56] INFO    | Handler.handle() -> 2.43s (attempt 1)
[02:04:58] INFO    | Validator: all_valid=True, approved=3 (2.34s)
[02:04:58] INFO    | Saved 3 questions to round 0 (total rounds: 1)
[02:04:58] INFO


Node                        Time (s)
  ContextAnalyzer              -
  IntentRouter                0.97s  #
  SessionManager               -
  ActionPlanner                -
  RAG                        29.14s  ##########################################################
  Handler:generate_quiz       2.43s  ####
  TOTAL                      34.91s

Response: 886 chars
